# NullVector + LangGraph: End-to-End Document QA Cookbook

This notebook walks through a complete, in-memory document question-answering pipeline that combines:

- **NullVector** — deterministic PDF acquisition, hierarchy building, and typed retrieval
- **LangGraph** — stateful agent orchestration with `MemorySaver` (in-memory checkpointing)
- **OpenRouter** — LLM backbone (gemini-3.1-flash-lite-preview)

## What you will build

```
PDF file
  └─► NullVector Acquisition  ──► CanonicalDocumentLedger
        └─► (optional) Tree Build  ──► HierarchyNodes
              └─► RetrievalCorpus  ──► InMemoryRetrievalIndex
                    └─► LangGraph ReAct Agent  ──► MemorySaver
                          ├─ search_document (NullVector retrieval tool)
                          └─ get_page_text    (NullVector page tool)
```

## Prerequisites

- Python 3.11+
- **PyMuPDF exactly 1.27.2** and **pypdf exactly 6.8.0** (NullVector version-locks these)
- The `nullvector` package installed from your project root


In [6]:
# ── 1. INSTALLATION ─────────────────────────────────────────────────────────
# NullVector version-locks PyMuPDF and pypdf. Install those first.
# If you already have nullvector installed from your project root,
# skip the last line.

!uv pip install -q \
    "pymupdf==1.27.2" \
    "pypdf==6.8.0" \
    "langgraph>=0.2" \
    "langchain-openai>=0.1" \
    "langchain-core>=0.2" \
    "fpdf2>=2.7" \
    "litellm>=1.40" \
    "pydantic>=2.5"

# Install nullvector from the project root (edit path as needed):
# %pip install -q -e /home/pruthvi/projects/NullVector

In [2]:
# ── 2. CONFIGURATION ────────────────────────────────────────────────────────
import os
import tempfile
from pathlib import Path

# OpenRouter credentials
OPENROUTER_API_KEY = "REDACTED_OPENROUTER_KEY"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# LangChain / direct API model name (OpenRouter uses the provider/model format WITHOUT
# the leading 'openrouter/' prefix that LiteLLM uses).
LANGCHAIN_MODEL = "google/gemini-2.0-flash-lite-preview-02-05"  # adjust if needed

# LiteLLM model name (for NullVector's own gateway, used in the bonus section)
LITELLM_MODEL = "openrouter/google/gemini-3.1-flash-lite-preview"  # as provided

# Set env vars so LangChain OpenAI client picks them up
os.environ["OPENAI_API_KEY"] = OPENROUTER_API_KEY  # LangChain reads this
os.environ["OPENAI_API_BASE"] = OPENROUTER_BASE_URL

# Artifact storage: NullVector writes parse artifacts to disk.
# We use a temp directory so everything is cleaned up on process exit.
ARTIFACT_DIR = tempfile.mkdtemp(prefix="nullvector_cookbook_")
print(f"Artifact directory: {ARTIFACT_DIR}")

Artifact directory: /tmp/nullvector_cookbook_vduk3hkl


In [ ]:
# # ── 3. CREATE A SAMPLE PDF ──────────────────────────────────────────────────
# # We generate a small multi-section PDF so the rest of the notebook
# # works without needing an external file.
# # Replace PDF_PATH with a path to your own document if preferred.

# from fpdf import FPDF

# SAMPLE_TEXT = {
#     "Chapter 1: Introduction to Machine Learning": (
#         "Machine learning is a branch of artificial intelligence that enables systems "
#         "to learn from data without being explicitly programmed. It relies on algorithms "
#         "that iteratively learn from data to improve their performance on a specific task. "
#         "The three main types are supervised learning, unsupervised learning, and "
#         "reinforcement learning. Supervised learning uses labelled training data, while "
#         "unsupervised learning finds hidden patterns without labels. Reinforcement learning "
#         "trains agents through reward signals from their environment."
#     ),
#     "Chapter 2: Neural Networks": (
#         "A neural network is a series of algorithms that attempt to recognise underlying "
#         "relationships in a set of data through a process that mimics the way the human brain "
#         "operates. Neural networks consist of layers: an input layer, one or more hidden "
#         "layers, and an output layer. Each layer contains units called neurons. The depth of "
#         "a network (number of hidden layers) gives rise to the term 'deep learning'. "
#         "Activation functions such as ReLU, sigmoid, and tanh introduce non-linearity. "
#         "Backpropagation is the standard algorithm for training neural networks by updating "
#         "weights using gradient descent."
#     ),
#     "Chapter 3: Transformers and Large Language Models": (
#         "The Transformer architecture, introduced in 'Attention Is All You Need' (2017), "
#         "revolutionised natural language processing. It relies entirely on self-attention "
#         "mechanisms instead of recurrence, enabling parallelism during training. Large "
#         "Language Models (LLMs) such as GPT, BERT, and Gemini are Transformer-based models "
#         "trained on enormous text corpora. They are used for text generation, classification, "
#         "question answering, and code generation. Instruction tuning and RLHF (Reinforcement "
#         "Learning from Human Feedback) align these models with human preferences."
#     ),
#     "Chapter 4: Evaluation and Benchmarks": (
#         "Evaluating machine learning models requires careful choice of metrics. For "
#         "classification tasks, common metrics are accuracy, precision, recall, and F1-score. "
#         "For regression, mean squared error (MSE) and mean absolute error (MAE) are standard. "
#         "Language model benchmarks include MMLU (Massive Multitask Language Understanding), "
#         "HellaSwag, TruthfulQA, and HumanEval for code. Overfitting is detected via "
#         "validation set performance. Regularisation techniques like dropout and weight decay "
#         "mitigate overfitting."
#     ),
#     "Chapter 5: Deployment and Production": (
#         "Deploying machine learning models in production introduces challenges around "
#         "latency, scalability, and data drift. Model serving frameworks include TorchServe, "
#         "TensorFlow Serving, and vLLM for LLMs. Quantisation reduces model size and speeds "
#         "up inference at the cost of minor accuracy loss. Monitoring pipelines detect drift "
#         "between training and production data distributions. Canary deployments and A/B "
#         "testing allow safe rollout of model updates. MLOps platforms such as MLflow, "
#         "Weights & Biases, and Kubeflow automate the ML lifecycle."
#     ),
# }

# pdf = FPDF()
# pdf.set_auto_page_break(auto=True, margin=15)

# for chapter_title, body_text in SAMPLE_TEXT.items():
#     pdf.add_page()
#     pdf.set_font("Helvetica", "B", 16)
#     pdf.multi_cell(0, 10, chapter_title)
#     pdf.ln(4)
#     pdf.set_font("Helvetica", size=11)
#     pdf.multi_cell(0, 6, body_text)

# PDF_PATH = os.path.join(ARTIFACT_DIR, "ml_overview.pdf")
# pdf.output(PDF_PATH)
# print(f"Sample PDF written to: {PDF_PATH}")
# print(f"File size: {Path(PDF_PATH).stat().st_size:,} bytes")

In [3]:
PDF_PATH = "/home/pruthvi/projects/NullVector/cookbook/903000608.pdf"

In [4]:
# ── 4. NULLVECTOR ACQUISITION ────────────────────────────────────────────────
# AcquisitionService reads the PDF with native PyMuPDF, extracts text blocks,
# line blocks, table artefacts, outlines, and writes a canonical ledger to disk.
# It also builds the canonical text substrate and projection view in one pass.

from nullvector.domain import AcquisitionRequest
from nullvector.ingest import acquire_document

ACQUISITION_RUN_ID = "cookbook-acq-001"

acquisition_request = AcquisitionRequest(
    source_path=PDF_PATH,
    acquisition_run_id=ACQUISITION_RUN_ID,
    artifact_root=ARTIFACT_DIR,
)

acq_manifest = acquire_document(acquisition_request)

print("=" * 60)
print("ACQUISITION COMPLETE")
print("=" * 60)
print(f"Document ID   : {acq_manifest.document_id}")
print(f"Page count    : {acq_manifest.page_count}")
print(f"Artifact root : {acq_manifest.artifact_root}")
print(f"Outline source: {acq_manifest.selected_outline_source}")

# The canonical text substrate path is needed for downstream steps
ACQ_MANIFEST_PATH = str(Path(acq_manifest.artifact_root) / "manifest.json")

'utf-16-be' codec can't decode byte 0x31 in position 16: truncated data
initial string:b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.41'
'utf-16-be' codec can't decode byte 0x44 in position 16: truncated data
initial string:b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.cD'
Removed unexpected destination b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.cD' from destination
Removed unexpected destination b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.41' from destination
'utf-16-be' codec can't decode byte 0x31 in position 16: truncated data
initial string:b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.41'
'utf-16-be' codec can't decode byte 0x44 in position 16: truncated data
initial string:b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.cD'
Removed unexpected destination b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.cD' from destination
Removed unexpected destination b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.41' from destination


ACQUISITION COMPLETE
Document ID   : 798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896
Page count    : 148
Artifact root : /tmp/nullvector_cookbook_vduk3hkl/cookbook-acq-001/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896
Outline source: pymupdf


In [7]:
# ── 5. (OPTIONAL) NULLVECTOR TREE BUILD ────────────────────────────────────
# The tree pipeline extracts a verified heading hierarchy and computes owned
# content spans per node.  It enriches the retrieval corpus with node-level
# text units that carry path / level / span metadata — useful for structural
# queries like "what does chapter 3 say about X?".
#
# This step runs deterministically (no LLM required by default).
# Set RUN_TREE_BUILD = False to skip and rely on page-level units only.

RUN_TREE_BUILD = True
TREE_MANIFEST_PATH = None  # will be set below if RUN_TREE_BUILD is True

if RUN_TREE_BUILD:
    from nullvector.domain import TreeBuildRequest
    from nullvector.tree import build_tree

    tree_request = TreeBuildRequest(
        acquisition_manifest_path=ACQ_MANIFEST_PATH,
        tree_run_id="cookbook-tree-001",
        summarize=False,  # set True + pass a gateway to generate LLM summaries
    )

    tree_manifest = build_tree(tree_request)
    TREE_MANIFEST_PATH = str(Path(tree_manifest.artifact_root) / "manifest.json")

    print("=" * 60)
    print("TREE BUILD COMPLETE")
    print("=" * 60)
    print(f"Committed nodes    : {tree_manifest.committed_node_count}")
    print(f"Unassigned spans   : {tree_manifest.unassigned_span_count}")
    print(f"Artifact root      : {tree_manifest.artifact_root}")
else:
    print("Tree build skipped — corpus will contain page-level units only.")

TREE BUILD COMPLETE
Committed nodes    : 3000
Unassigned spans   : 1
Artifact root      : /tmp/nullvector_cookbook_vduk3hkl/cookbook-acq-001/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896/tree/cookbook-tree-001


In [ ]:
# ── 6. BUILD RETRIEVAL CORPUS AND IN-MEMORY INDEX ───────────────────────────
# RetrievalCorpusBuilder assembles all evidence units:
#   • PAGE_TEXT       — full page text (always present)
#   • TABLE           — detected tabular regions
#   • VISUAL          — embedded images
#   • NODE_TEXT       — per-node owned text (if tree was built)
#   • NODE_SUMMARY    — LLM-generated summaries (if summarize=True was used)
#   • UNASSIGNED_SPAN — pages not owned by any verified node
#
# InMemoryRetrievalIndex builds posting lists in RAM for instant BM25-like
# scoring — no vector store, no embeddings required.

from nullvector.retrieval import (
    InMemoryRetrievalIndex,
    QueryPlanner,
    RetrievalCorpusBuilder,
    RetrievalRanker,
    RetrievalService,
    load_retrieval_corpus,
)

builder = RetrievalCorpusBuilder()
retrieval_manifest = builder.build(
    acquisition_manifest_path=ACQ_MANIFEST_PATH,
    tree_manifest_path=TREE_MANIFEST_PATH,  # None → page-only corpus
)

# Load corpus from the written JSON artefact
corpus = load_retrieval_corpus(retrieval_manifest.corpus_path)

# Build the in-memory index — this is cheap and fast
index = InMemoryRetrievalIndex(corpus)

print("=" * 60)
print("RETRIEVAL CORPUS READY")
print("=" * 60)
print(f"Total units  : {retrieval_manifest.unit_count}")
print(f"Document ID  : {corpus.document_id}")
from collections import Counter

counts = Counter(u.unit_type.value for u in corpus.units)
for unit_type, n in sorted(counts.items()):
    print(f"  {unit_type:<22}: {n}")

# Set up the retrieval service (planner + ranker + index wrapper)
planner = QueryPlanner()
ranker = RetrievalRanker()
service = RetrievalService(planner, ranker)

# Quick sanity check
hits = service.search(corpus=corpus, query="neural network activation functions", limit=3)
print("\nTop-3 hits for sanity-check query:")
for h in hits:
    snippet = (h.unit.text or "")[:120].replace("\n", " ")
    print(
        f"  [{h.unit.unit_type.value}] page {h.unit.page_span.start_page + 1} "
        f"score={h.score:.1f} — {snippet}..."
    )

---
## LangGraph Agent

We now build a **ReAct-style** agent that:

1. Receives a user message in `State.messages`
2. Calls the LLM; if it emits tool calls, routes to the tool executor
3. Injects NullVector retrieval results back as `ToolMessage`s
4. Loops until the model produces a final answer

State is persisted with `MemorySaver` so multi-turn conversations work out of the box — all in-memory, no database needed.

```
          ┌──────────────┐
  START ──► call_model   ├─── tool_calls? ──YES──► execute_tools ──┐
          └──────────────┘                                          │
                  ▲                                                 │
                  └─────────────────────────────────────────────────┘
                  │
                  └── no tool_calls ──► END
```

In [ ]:
# ── 7. STATE SCHEMA ─────────────────────────────────────────────────────────
from typing import Annotated, TypedDict

from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages


class AgentState(TypedDict):
    """The entire conversation is stored here.
    `add_messages` is a reducer that appends new messages rather than
    overwriting — this is what enables multi-turn memory.
    """

    messages: Annotated[list[BaseMessage], add_messages]


print("State schema defined: AgentState with add_messages reducer.")

In [ ]:
# ── 8. NULLVECTOR RETRIEVAL TOOLS ───────────────────────────────────────────
# We expose two tools to the LLM:
#   search_document — ranked retrieval over all unit types
#   get_page_text   — fetch the raw text of a specific page

from langchain_core.tools import tool


@tool
def search_document(query: str, limit: int = 5) -> str:
    """Search the document for content relevant to the query.
    Returns ranked text excerpts with page numbers and unit type.
    Use this for any factual question about the document.
    Args:
        query: Natural language search query.
        limit: Maximum number of results to return (default 5).
    """
    hits = service.search(corpus=corpus, query=query, limit=max(1, min(int(limit), 10)))
    if not hits:
        return "No relevant content found in the document for that query."

    parts = []
    for i, hit in enumerate(hits, 1):
        page_label = hit.unit.page_span.start_page + 1
        title = hit.unit.title or "(untitled)"
        text = (hit.unit.text or "").strip()
        # Trim long excerpts to keep context window manageable
        excerpt = text[:600] + ("..." if len(text) > 600 else "")
        unit_type = hit.unit.unit_type.value
        parts.append(
            f"[Result {i} | page {page_label} | type={unit_type} | score={hit.score:.1f}]\n"
            f"Title: {title}\n"
            f"{excerpt}"
        )
    return "\n\n".join(parts)


@tool
def get_page_text(page_number: int) -> str:
    """Retrieve the complete raw text of a specific page (1-indexed).
    Useful when you need to read an entire page in full rather than
    a ranked excerpt.
    Args:
        page_number: 1-indexed page number.
    """
    page_index = max(0, int(page_number) - 1)
    pages_by_index = {
        u.page_span.start_page: u.text
        for u in corpus.units
        if u.unit_type.value == "page_text" and u.page_span.start_page == u.page_span.end_page
    }
    text = pages_by_index.get(page_index)
    if text is None:
        return f"Page {page_number} not found. Document has {len(pages_by_index)} pages."
    return f"[Page {page_number}]\n{text.strip()}"


TOOLS = [search_document, get_page_text]
TOOL_MAP = {t.name: t for t in TOOLS}

print("Tools registered:", [t.name for t in TOOLS])

In [ ]:
# ── 9. LLM SETUP ─────────────────────────────────────────────────────────────
# Using langchain-openai with the OpenRouter base URL.
# The OpenRouter API is OpenAI-compatible; just swap the base_url and api_key.

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY,
    model=LANGCHAIN_MODEL,
    temperature=0.0,
)

# Bind tools so the model can emit structured tool_call messages
llm_with_tools = llm.bind_tools(TOOLS)

print(f"LLM ready: {LANGCHAIN_MODEL} via OpenRouter")
print(f"Tools bound: {[t.name for t in TOOLS]}")

In [ ]:
# ── 10. BUILD THE LANGGRAPH AGENT ────────────────────────────────────────────
from langchain_core.messages import AIMessage, SystemMessage, ToolMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph

SYSTEM_PROMPT = (
    "You are a helpful document assistant. You have access to tools that let you "
    "search the document and retrieve page content. Always ground your answers in "
    "evidence from the document. Cite page numbers when available."
)


# ── Node: call_model ────────────────────────────────────────────────────────
def call_model(state: AgentState) -> dict:
    """Invoke the LLM with the full conversation history."""
    messages = state["messages"]
    # Prepend system message if this is the first turn
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + list(messages)
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


# ── Node: execute_tools ──────────────────────────────────────────────────────
def execute_tools(state: AgentState) -> dict:
    """Execute all tool calls in the last AI message and return ToolMessages."""
    last_message: AIMessage = state["messages"][-1]
    tool_messages = []
    for tool_call in last_message.tool_calls:
        tool_fn = TOOL_MAP.get(tool_call["name"])
        if tool_fn is None:
            result = f"Unknown tool: {tool_call['name']}"
        else:
            try:
                result = tool_fn.invoke(tool_call["args"])
            except Exception as exc:
                result = f"Tool error: {exc}"
        tool_messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"],
                name=tool_call["name"],
            )
        )
    return {"messages": tool_messages}


# ── Edge: should_continue ────────────────────────────────────────────────────
def should_continue(state: AgentState) -> str:
    """Route to 'tools' if there are pending tool calls, else to END."""
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and getattr(last, "tool_calls", None):
        return "tools"
    return END


# ── Assemble the graph ───────────────────────────────────────────────────────
builder_graph = StateGraph(AgentState)
builder_graph.add_node("agent", call_model)
builder_graph.add_node("tools", execute_tools)
builder_graph.set_entry_point("agent")
builder_graph.add_conditional_edges(
    "agent",
    should_continue,
    {"tools": "tools", END: END},
)
builder_graph.add_edge("tools", "agent")

# ── Compile with in-memory checkpointer ─────────────────────────────────────
memory = MemorySaver()  # all state is kept in Python dicts — no DB
app = builder_graph.compile(checkpointer=memory)

print("LangGraph compiled with MemorySaver (in-memory checkpointing).")

In [ ]:
# ── 11. VISUALISE THE GRAPH (ASCII / Mermaid) ────────────────────────────────
# LangGraph can render a Mermaid diagram. If graphviz/Pillow are installed
# the PNG version can also be displayed inline in Jupyter.

try:
    from IPython.display import Image, display

    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    # Fallback: print the Mermaid source
    print(app.get_graph().draw_mermaid())

In [ ]:
# ── 12. SINGLE-TURN QUERY DEMO ───────────────────────────────────────────────
# Each conversation is identified by a `thread_id` config.
# Using the same thread_id later will recall the full history from MemorySaver.

from langchain_core.messages import HumanMessage


def ask(question: str, thread_id: str = "demo-thread-1", verbose: bool = True) -> str:
    """Send a message to the agent and return its final answer."""
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke(
        {"messages": [HumanMessage(content=question)]},
        config=config,
    )
    final_msg = result["messages"][-1]
    answer = final_msg.content

    if verbose:
        # Show what happened step by step
        print(f"Q: {question}")
        print("-" * 60)
        for msg in result["messages"]:
            role = type(msg).__name__
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                calls = [{"name": tc["name"], "args": tc["args"]} for tc in msg.tool_calls]
                print(f"[{role}] → tool_calls: {calls}")
            elif isinstance(msg, ToolMessage):
                preview = str(msg.content)[:150].replace("\n", " ")
                print(f"[ToolMessage:{msg.name}] → {preview}...")
            elif isinstance(msg, HumanMessage):
                pass  # already printed above
            else:
                if msg.content:
                    print(f"[{role}] {msg.content}")
    return answer


# First question on a fresh thread
answer1 = ask(
    "What are the three main types of machine learning?",
    thread_id="thread-001",
)

In [ ]:
# ── 13. MULTI-TURN CONVERSATION DEMO ─────────────────────────────────────────
# Because we use the SAME thread_id, the agent remembers previous turns.
# The MemorySaver replays the full message history on every invocation.

print("=" * 60)
print("MULTI-TURN CONVERSATION (thread: thread-002)")
print("=" * 60)

THREAD = "thread-002"

# Turn 1
print("\n── Turn 1 ──")
ask("What is a Transformer and why does it matter for LLMs?", thread_id=THREAD)

# Turn 2 — follow-up referring to Turn 1 context
print("\n── Turn 2 ──")
ask(
    "You mentioned attention. How does that compare to what backpropagation does?", thread_id=THREAD
)

# Turn 3 — structural / page-level question
print("\n── Turn 3 ──")
ask("Which chapter covers deployment and production?", thread_id=THREAD)

# Inspect saved state directly from MemorySaver
snapshot = app.get_state({"configurable": {"thread_id": THREAD}})
n_msgs = len(snapshot.values.get("messages", []))
print(f"\nMessages stored in MemorySaver for thread '{THREAD}': {n_msgs}")

In [ ]:
# ── 14. STREAMING OUTPUT ─────────────────────────────────────────────────────
# LangGraph supports streaming graph events. Here we stream token-by-token output
# AND graph step events so you can see which node is executing.


config = {"configurable": {"thread_id": "thread-streaming"}}
user_input = {"messages": [HumanMessage(content="Summarise the chapter on evaluation metrics.")]}

print("Streaming graph events:\n")
for event in app.stream(user_input, config=config, stream_mode="values"):
    last_msg = event["messages"][-1]
    role = type(last_msg).__name__
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        print(f"  [{role}] emitting tool_calls → {[tc['name'] for tc in last_msg.tool_calls]}")
    elif isinstance(last_msg, ToolMessage):
        print(f"  [ToolMessage:{last_msg.name}] returned {len(str(last_msg.content))} chars")
    else:
        preview = str(last_msg.content)[:200].replace("\n", " ")
        print(f"  [{role}]: {preview}{'...' if len(str(last_msg.content)) > 200 else ''}")

print("\nStream complete.")

---
## Bonus: Using the NullVector LLM Gateway inside a LangGraph Node

NullVector ships its own typed LLM gateway (`GatewayService`) that handles structured
output, retries, and audit logging.  You can call it from inside a custom LangGraph node
to generate node summaries or run document repair on-the-fly.

> **Note on LiteLLM:** `LiteLLMSDKAdapter` internally calls `litellm.responses()`, which
> is LiteLLM's Responses-API endpoint. OpenRouter supports this via a compatibility
> shim, but you may need `litellm >= 1.50` for full support.  If you hit issues, see the
> custom adapter pattern in the cell below.

In [ ]:
# ── 15. NULLVECTOR GATEWAY CONFIGURATION ────────────────────────────────────
# Configure NullVector's own LLM gateway to use OpenRouter via LiteLLM.

from nullvector.llm import GatewayConfig, GatewayService
from nullvector.llm.types import GatewayAuditConfig, LiteLLMProviderConfig

nv_gateway_config = GatewayConfig(
    provider=LiteLLMProviderConfig(
        model=LITELLM_MODEL,  # "openrouter/google/gemini-3.1-flash-lite-preview"
        api_key=OPENROUTER_API_KEY,
        api_base="https://openrouter.ai/api/v1",
    ),
    timeout_seconds=30.0,
    audit=GatewayAuditConfig(
        persist_root=None,  # set to a directory path to save audit records
        capture_raw_request=False,
        capture_raw_response=False,
    ),
)

nv_gateway = GatewayService(nv_gateway_config)
print("NullVector GatewayService configured.")

# ── Example: call the gateway directly with a structured output model ────────
from nullvector.domain import NonEmptyStr
from nullvector.domain.common import StrataModel
from nullvector.llm.types import GatewayRequest, LLMMessage, LLMRole


class OneLinerSummary(StrataModel):
    """A single-sentence summary of a document section."""

    sentence: NonEmptyStr


def summarise_via_nv_gateway(text: str) -> str:
    """Use NullVector's gateway to summarise a text excerpt."""
    request = GatewayRequest[OneLinerSummary](
        operation_name="one_liner_summary",
        messages=(
            LLMMessage(
                role=LLMRole.SYSTEM,
                content="You are a precise technical summariser. Respond only with the schema.",
            ),
            LLMMessage(
                role=LLMRole.USER,
                content=f"Produce a one-sentence summary of:\n\n{text[:800]}",
            ),
        ),
        response_model=OneLinerSummary,
        temperature=0.0,
    )
    result = nv_gateway.invoke(request)
    return result.output.sentence


# Test: summarise the first page
first_page_units = [
    u for u in corpus.units if u.unit_type.value == "page_text" and u.page_span.start_page == 0
]
if first_page_units:
    page_text = first_page_units[0].text or ""
    try:
        summary = summarise_via_nv_gateway(page_text)
        print(f"One-liner summary of page 1:\n  {summary}")
    except Exception as exc:
        print(f"Gateway call failed (check LiteLLM/OpenRouter compat): {exc}")

In [ ]:
# ── 16. LANGGRAPH NODE USING NULLVECTOR GATEWAY ──────────────────────────────
# We show how to add a dedicated 'summarise_pages' node to a LangGraph graph
# that uses the NullVector gateway to generate structured summaries, then
# injects them back into the conversation as a SystemMessage.

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph


class EnrichedState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    summaries_injected: bool


def summarise_pages_node(state: EnrichedState) -> dict:
    """Pre-compute page summaries with the NullVector gateway and inject them
    into the conversation context before the first LLM call."""
    if state.get("summaries_injected"):
        return {}  # already done — skip

    page_units = [u for u in corpus.units if u.unit_type.value == "page_text" and u.text]
    summaries = []
    for unit in page_units[:3]:  # limit to first 3 pages for demo speed
        page_num = unit.page_span.start_page + 1
        try:
            s = summarise_via_nv_gateway(unit.text)
            summaries.append(f"Page {page_num}: {s}")
        except Exception:
            summaries.append(f"Page {page_num}: (summary unavailable)")

    context_msg = SystemMessage(
        content="Document page summaries (pre-computed):\n" + "\n".join(summaries)
    )
    return {"messages": [context_msg], "summaries_injected": True}


# Assemble an enriched graph with a pre-summarisation step
enriched_graph = StateGraph(EnrichedState)
enriched_graph.add_node("summarise", summarise_pages_node)
enriched_graph.add_node("agent", call_model)
enriched_graph.add_node("tools", execute_tools)
enriched_graph.set_entry_point("summarise")
enriched_graph.add_edge("summarise", "agent")
enriched_graph.add_conditional_edges(
    "agent",
    should_continue,
    {"tools": "tools", END: END},
)
enriched_graph.add_edge("tools", "agent")

enriched_memory = MemorySaver()
enriched_app = enriched_graph.compile(checkpointer=enriched_memory)

print("Enriched graph (with NullVector gateway pre-summarisation) compiled.")

# Run a query through the enriched graph
print("\nRunning query through enriched graph...")
cfg = {"configurable": {"thread_id": "enriched-001"}}
result = enriched_app.invoke(
    {
        "messages": [HumanMessage(content="What does chapter 1 discuss?")],
        "summaries_injected": False,
    },
    config=cfg,
)
print("\nAnswer:", result["messages"][-1].content)

In [ ]:
# ── 17. EXPORT CORPUS TO LANGCHAIN DOCUMENTS ────────────────────────────────
# NullVector includes edge exporters for LangChain and LlamaIndex.
# This lets you slot NullVector's structured corpus into any LangChain RAG pipeline.

import json

from nullvector.domain import NodeCard
from nullvector.export import to_langchain_documents

# Load node cards if tree was built
lc_docs = []
if TREE_MANIFEST_PATH:
    tree_manifest_obj = __import__("json")  # just to load JSON below
    from nullvector.domain import NodeCard

    cards_path = None
    with open(TREE_MANIFEST_PATH) as f:
        tm = json.load(f)
    cards_path = tm.get("node_cards_path")

    if cards_path and Path(cards_path).exists():
        with open(cards_path) as f:
            raw_cards = json.load(f)
        node_cards = tuple(NodeCard.model_validate(c) for c in raw_cards)
        lc_docs = to_langchain_documents(node_cards)
        print(f"Exported {len(lc_docs)} LangChain documents from NullVector node cards.")
        if lc_docs:
            first = lc_docs[0]
            print("\nFirst document preview:")
            print(f"  page_content: {str(first.page_content)[:200]}")
            print(f"  metadata keys: {list(first.metadata.keys())}")
    else:
        print("Node cards not found — skipping LangChain export.")
else:
    print("Tree build was skipped — no node cards to export.")
    print("(Set RUN_TREE_BUILD = True in cell 5 to enable this.)")

In [ ]:
# ── 18. INSPECT IN-MEMORY STATE ──────────────────────────────────────────────
# MemorySaver stores all thread states in a dict-like structure in RAM.
# Here we show how to list threads, replay history, and manually inject state.

# List all threads currently in MemorySaver
# (access the underlying store — implementation detail but useful for debugging)
print("Threads in MemorySaver:")
for thread_id in ["thread-001", "thread-002", "thread-streaming"]:
    cfg = {"configurable": {"thread_id": thread_id}}
    try:
        snap = app.get_state(cfg)
        n = len(snap.values.get("messages", []))
        print(f"  {thread_id}: {n} messages")
    except Exception:
        print(f"  {thread_id}: (no state)")

# Replay thread-001 history
print("\nFull message history for thread-001:")
snap = app.get_state({"configurable": {"thread_id": "thread-001"}})
for msg in snap.values.get("messages", []):
    role = type(msg).__name__.replace("Message", "")
    preview = str(msg.content)[:100].replace("\n", " ") if msg.content else "(tool_calls)"
    print(f"  [{role:12s}] {preview}")

In [ ]:
# ── 19. CLEANUP ──────────────────────────────────────────────────────────────
# Remove all NullVector artifacts from the temp directory.
# Skip this cell if you want to inspect the on-disk artefacts.

import shutil

shutil.rmtree(ARTIFACT_DIR, ignore_errors=True)
print(f"Cleaned up: {ARTIFACT_DIR}")

---
## What to try next

| Goal | How |
|------|-----|
| **Use your own PDF** | Change `PDF_PATH` in cell 4 to your file path |
| **Generate node summaries** | Set `summarize=True` in `TreeBuildRequest` and pass `gateway=nv_gateway` |
| **Visual enrichment** | Use `enrich_visual_region` from `nullvector.llm` with a `GatewayService` |
| **Persist across sessions** | Swap `MemorySaver` for `SqliteSaver` or `PostgresSaver` from `langgraph-checkpoint-*` |
| **Add more tools** | Wrap `RetrievalQAService` or `to_langchain_documents` as LangGraph tools |
| **Custom planner** | Subclass `QueryPlanner` to add domain-specific filters |
| **Export to LlamaIndex** | Use `nullvector.export.to_llamaindex_nodes` in place of `to_langchain_documents` |
| **Stream tokens** | Replace `app.invoke` with `app.astream_events` (async) for real-time output |

### Key NullVector classes reference

```python
from nullvector.ingest    import acquire_document, AcquisitionService
from nullvector.tree      import build_tree, TreeBuildRequest
from nullvector.retrieval import RetrievalCorpusBuilder, InMemoryRetrievalIndex
from nullvector.retrieval import RetrievalService, QueryPlanner, RetrievalRanker
from nullvector.retrieval import RetrievalQAService
from nullvector.llm       import GatewayService, GatewayConfig, enrich_visual_region
from nullvector.export    import to_langchain_documents, to_llamaindex_nodes
from nullvector.observability import EventBus, JsonLoggerSubscriber
```